<a href="https://colab.research.google.com/github/sashi-does-cloud/llm-agentic-rag/blob/main/groq_weather_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Weather Agent with Groq Tool Calling

This notebook demonstrates how to implement an autonomous function-calling loop using the **Groq API** (`llama-3.1-8b-instant`) and the **OpenWeatherMap API** to fetch real-time weather information.

### Required Colab Secrets
1. `GROQ_API_KEY`(https://console.groq.com/keys)
2. `OPEN_WEATHER_API_KEY`(https://home.openweathermap.org/api_keys)

In [1]:
!pip install groq requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.6 MB/s eta 0:00:00


## 1. Imports and Client Initialization

In [2]:
import os
import json
import requests
from groq import Groq
from google.colab import userdata
from pprint import pprint

api_key = userdata.get("GROQ_API_KEY")
weather_api_key = userdata.get("OPEN_WEATHER_API_KEY")

client = Groq(api_key=api_key)

## 2. Define the Tool Specifications and Local Python Functions

We specify the tool description so the LLM understands when and how to call it. We also define the corresponding local Python function `get_temperature` to perform the actual weather lookup.

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_temperature",
            "description": "Get temperature for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

def get_temperature(city):
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={weather_api_key}&units=metric"
    response = requests.get(url)
    data = response.json()
    if 'main' in data:
        return data['main']
    else:
        return {"error": data.get("message", "City not found")}

## 3. Start Conversation and Call the LLM with Tools

In [6]:
llm_messages = [
    {
        "role": "user",
        "content": "current temperature at vijayawada"
    }
]

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    tools=tools,
    tool_choice="auto",
    messages=llm_messages
)

assistant_message = response.choices[0].message

## 4. Handle Tool Executions and Append to Conversation

Here we execute the tool requested by the model, capture its result, and append both the tool call request (from the assistant) and the tool result (from the system) in the correct chronological order.

In [7]:
weather_data = None
city = None

if assistant_message.tool_calls:
    llm_messages.append(assistant_message)

    tool_call = assistant_message.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    city = args.get('city')

    print(f"Executing local function for: {city}")
    weather_data = get_temperature(city)
    print(f"Weather API Result: {weather_data}")

    llm_messages.append({
        'role': 'tool',
        'tool_call_id': tool_call.id,
        'content': json.dumps(weather_data)
    })
else:
    print("No tool execution was requested by the model.")

Executing local function for: Vijayawada
Weather API Result: {'temp': 40.97, 'feels_like': 46.64, 'temp_min': 40.97, 'temp_max': 40.97, 'pressure': 1002, 'humidity': 33, 'sea_level': 1002, 'grnd_level': 998}


## 5. Send Tool Output back to Groq for the Final Answer

In [8]:
print("Current conversation history:")
pprint(llm_messages, indent=3)

final_response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=llm_messages
)

print("\n--- FINAL RESPONSE FROM MODEL ---")
print(final_response.choices[0].message.content)

Current conversation history:
[  {'content': 'current temperature at vijayawada', 'role': 'user'},
   ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='tkrtgand1', function=Function(arguments='{"city":"Vijayawada"}', name='get_temperature'), type='function')]),
   {  'content': '{"temp": 40.97, "feels_like": 46.64, "temp_min": 40.97, '
                 '"temp_max": 40.97, "pressure": 1002, "humidity": 33, '
                 '"sea_level": 1002, "grnd_level": 998}',
      'role': 'tool',
      'tool_call_id': 'tkrtgand1'}]

--- FINAL RESPONSE FROM MODEL ---
The current temperature at Vijayawada is 40.97°C (105.5°F). Please note that the temperature may vary depending on the source and the time of the request.

sashi
